In [7]:
#  Importing Basic essential libraries
import pandas as pd
import numpy as np
import re

In [8]:
#Loading the Dataset
df = pd.read_csv("/content/emails.csv",
                 engine="python",
                 on_bad_lines="skip",
                 encoding="latin1")

df.head()

,file,message
0,allen-p/_sent_mail/1.,Message-ID: <18782981.1075855378110.JavaMail.e...
1,allen-p/_sent_mail/10.,Message-ID: <15464986.1075855378456.JavaMail.e...
2,allen-p/_sent_mail/100.,Message-ID: <24216240.1075855687451.JavaMail.e...
3,allen-p/_sent_mail/1000.,Message-ID: <13505866.1075863688222.JavaMail.e...
4,allen-p/_sent_mail/1001.,Message-ID: <30922949.1075863688243.JavaMail.e...


In [9]:
#Inspect Data to check total rows,columns & datatype
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.info()

Shape: (843222, 2)
Columns: ['file', 'message']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 843222 entries, 0 to 843221
Data columns (total 2 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   file     842605 non-null  object
 1   message  554748 non-null  object
dtypes: object(2)
memory usage: 12.9+ MB


In [ ]:
#Data Cleaning
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words("english"))

def clean_text(t):
    t = t.lower()
    t = re.sub(r"http\S+","", t)
    t = re.sub(r"[^a-z ]"," ", t)
    t = re.sub(r"\s+"," ", t).strip()          # remove extra spaces

    # remove stopwords
    t = " ".join([word for word in t.split() if word not in stop_words])

    return t

X_train = X_train.apply(clean_text)
X_test = X_test.apply(clean_text)

In [11]:
#Create Labels on Dataset (Temporary spam/ham)
df["label"] = np.random.choice(["spam", "ham"], len(df))
df.head()

,file,message,clean_message,label
0,allen-p/_sent_mail/1.,Message-ID: <18782981.1075855378110.JavaMail.e...,Message-ID: <18782981.1075855378110.JavaMail.e...,spam
1,allen-p/_sent_mail/10.,Message-ID: <15464986.1075855378456.JavaMail.e...,Message-ID: <15464986.1075855378456.JavaMail.e...,ham
2,allen-p/_sent_mail/100.,Message-ID: <24216240.1075855687451.JavaMail.e...,Message-ID: <24216240.1075855687451.JavaMail.e...,spam
3,allen-p/_sent_mail/1000.,Message-ID: <13505866.1075863688222.JavaMail.e...,Message-ID: <13505866.1075863688222.JavaMail.e...,ham
4,allen-p/_sent_mail/1001.,Message-ID: <30922949.1075863688243.JavaMail.e...,Message-ID: <30922949.1075863688243.JavaMail.e...,ham


In [12]:
#Split Train & Test Data
from sklearn.model_selection import train_test_split

X = df["clean_message"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

len(X_train), len(X_test)

(674577, 168645)

In [ ]:
#TF IDF VECTORIZATION
#Convert Text to Numerical Vector by TF IDF
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words="english")

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [ ]:
#Importing  Models Library
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

# Create model objects
nb = MultinomialNB()
lr = LogisticRegression(max_iter=1000)
dt = DecisionTreeClassifier()

# Train models
nb.fit(X_train_tfidf, y_train)
lr.fit(X_train_tfidf, y_train)
dt.fit(X_train_tfidf, y_train)

print("Models trained successfully!")

In [ ]:
#Evaluate (Accuracy +F1+ Confusion Matrix)
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

models = [nb, lr, dt]
names = ["Naive Bayes", "Logistic Regression", "Decision Tree"]

for name, model in zip(names, models):
    print("\n==============================")
    print("MODEL:", name)
    print("==============================")

    y_pred = model.predict(X_test_tfidf)

    print("Accuracy:", accuracy_score(y_test, y_pred))

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    cm = confusion_matrix(y_test, y_pred)
    print("Confusion Matrix:")
    print(cm)

    TN, FP, FN, TP = cm.ravel()
    print("\nTN:", TN)
    print("FP:", FP)
    print("FN:", FN)
    print("TP:", TP)

In [ ]:
#Predicting the Email
def predict_email(text, model):
    text_tfidf = tfidf.transform([text])
    return model.predict(text_tfidf)[0]

sample = input("Enter email text: ")
print("Prediction:", predict_email(sample, lr))